# Silver Transform — Energy Grid (Lakehouse)

Cleans and enriches Bronze tables into Silver Delta tables.
This is the Lakehouse variant (for ML scenarios).

In [ ]:
from pyspark.sql.functions import (
    col, current_timestamp, to_date, hour, when, abs as spark_abs,
    avg, stddev
)
from pyspark.sql.window import Window

print('Silver transform starting')

In [ ]:
# Silver grid sensors — clean, validate, add derived columns
sensors = spark.read.table('bronze_grid_sensors')

w = Window.partitionBy('substation_id')
silver_sensors = (
    sensors
    .filter(col('voltage_v').isNotNull() & col('frequency_hz').isNotNull())
    .withColumn('sensor_date', to_date('timestamp'))
    .withColumn('sensor_hour', hour('timestamp'))
    .withColumn('voltage_deviation', spark_abs(col('voltage_v') - 230.0))
    .withColumn('freq_deviation', spark_abs(col('frequency_hz') - 50.0))
    .withColumn('volt_mean', avg('voltage_v').over(w))
    .withColumn('volt_std', stddev('voltage_v').over(w))
    .withColumn('voltage_anomaly',
        when(spark_abs(col('voltage_v') - col('volt_mean')) > 2 * col('volt_std'), True).otherwise(False)
    )
    .withColumn('silver_timestamp', current_timestamp())
    .drop('volt_mean', 'volt_std')
)

silver_sensors.write.mode('overwrite').format('delta').saveAsTable('silver_grid_sensors')
print(f'Silver grid sensors: {silver_sensors.count()} rows')

In [ ]:
# Silver power events — clean, validate
events = spark.read.table('bronze_power_events')

silver_events = (
    events
    .filter(col('event_type').isNotNull())
    .withColumn('event_date', to_date('timestamp'))
    .withColumn('event_hour', hour('timestamp'))
    .withColumn('is_outage', when(col('event_type') == 'outage', True).otherwise(False))
    .withColumn('silver_timestamp', current_timestamp())
)

silver_events.write.mode('overwrite').format('delta').saveAsTable('silver_power_events')
print(f'Silver power events: {silver_events.count()} rows')

In [ ]:
# Silver renewable generation — clean
renewable = spark.read.table('bronze_renewable_generation')

silver_renewable = (
    renewable
    .filter(col('generation_mw').isNotNull())
    .withColumn('gen_date', to_date('timestamp'))
    .withColumn('silver_timestamp', current_timestamp())
)

silver_renewable.write.mode('overwrite').format('delta').saveAsTable('silver_renewable_generation')
print(f'Silver renewable generation: {silver_renewable.count()} rows')

In [ ]:
# Optimize
spark.sql('OPTIMIZE silver_grid_sensors')
spark.sql('OPTIMIZE silver_power_events')
spark.sql('OPTIMIZE silver_renewable_generation')
print('Silver transform complete')